In [ ]:
# ============================================================
# LANGGRAPH + MCP MASTER REVISION
#
# Covered:
#
# 1. StateGraph
# 2. State
# 3. Reducers
# 4. Nodes
# 5. Edges
# 6. START / END
# 7. Memory / Checkpointer
# 8. Tool calling
# 9. ReAct agent
# 10. HITL
# 11. FastMCP server - stdio
# 12. FastMCP server - HTTP
# 13. MCP client
# 14. Multi-server MCP client
#
# ============================================================


# ============================================================
# PART 1 — BASIC LANGGRAPH
# ============================================================

from typing import TypedDict

from langgraph.graph import StateGraph, START, END


# ------------------------------------------------------------
# 1. STATE
# ------------------------------------------------------------

class State(TypedDict):
    message: str


# State is the data that flows through the graph.
#
# Think:
#
#       State
#         ↓
#     ┌───────┐
#     │ graph │
#     └───────┘
#         ↓
#    updated State
#
#
# A node:
#
#     State → State
#
#
# ------------------------------------------------------------
# 2. NODE
# ------------------------------------------------------------

def hello_node(state: State):
    return {
        "message": "Hello " + state["message"]
    }


# A node is simply a function that:
#
#     receives state
#          ↓
#     does something
#          ↓
#     returns state updates


# ------------------------------------------------------------
# 3. BUILD GRAPH
# ------------------------------------------------------------

builder = StateGraph(State)


# Add node

builder.add_node(
    "hello",
    hello_node
)


# ------------------------------------------------------------
# 4. EDGES
# ------------------------------------------------------------

builder.add_edge(
    START,
    "hello"
)

builder.add_edge(
    "hello",
    END
)


# Graph:
#
#     START
#       ↓
#     hello
#       ↓
#      END


# ------------------------------------------------------------
# 5. COMPILE
# ------------------------------------------------------------

graph = builder.compile()


# ------------------------------------------------------------
# 6. INVOKE
# ------------------------------------------------------------

result = graph.invoke(
    {
        "message": "Pratyush"
    }
)

print(result)


# ============================================================
# PART 2 — MULTIPLE NODES
# ============================================================

class MathState(TypedDict):
    number: int
    result: int


def double_node(state: MathState):
    return {
        "result": state["number"] * 2
    }


def add_ten_node(state: MathState):
    return {
        "result": state["result"] + 10
    }


builder = StateGraph(MathState)

builder.add_node("double", double_node)
builder.add_node("add_ten", add_ten_node)

builder.add_edge(START, "double")
builder.add_edge("double", "add_ten")
builder.add_edge("add_ten", END)

graph = builder.compile()

result = graph.invoke(
    {
        "number": 5,
        "result": 0
    }
)

print(result)


# Execution:
#
#     State
#       ↓
#     double
#       ↓
#     add_ten
#       ↓
#     END
#
#
# State changes:
#
#     number=5
#     result=0
#
#        ↓
#
#     double
#     result=10
#
#        ↓
#
#     add_ten
#     result=20


# ============================================================
# PART 3 — REDUCERS
# ============================================================

from typing import Annotated
from operator import add


class MessageState(TypedDict):
    messages: Annotated[list[str], add]


# Without a reducer:
#
#     node returns:
#
#     {"messages": ["hello"]}
#
#     State value may be replaced.
#
#
# With:
#
#     Annotated[list, add]
#
# the reducer says:
#
#     OLD VALUE + NEW VALUE
#
#
# Example:
#
#     old:
#     ["hello"]
#
#     node returns:
#     ["world"]
#
#     reducer:
#     add
#
#     result:
#     ["hello", "world"]


def node_one(state: MessageState):
    return {
        "messages": ["hello"]
    }


def node_two(state: MessageState):
    return {
        "messages": ["world"]
    }


builder = StateGraph(MessageState)

builder.add_node("one", node_one)
builder.add_node("two", node_two)

builder.add_edge(START, "one")
builder.add_edge("one", "two")
builder.add_edge("two", END)

graph = builder.compile()


result = graph.invoke(
    {
        "messages": []
    }
)

print(result)


# ============================================================
# PART 4 — MEMORY / CHECKPOINTING
# ============================================================

from langgraph.checkpoint.memory import InMemorySaver


class ChatState(TypedDict):
    messages: Annotated[list[str], add]


def chat_node(state: ChatState):
    return {
        "messages": ["AI: I remember this conversation."]
    }


builder = StateGraph(ChatState)

builder.add_node(
    "chat",
    chat_node
)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)


checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# A thread identifies one conversation.

config = {
    "configurable": {
        "thread_id": "user-123"
    }
}


graph.invoke(
    {
        "messages": ["Human: My name is Pratyush."]
    },
    config=config
)


# Later:

graph.invoke(
    {
        "messages": ["Human: What is my name?"]
    },
    config=config
)


# Same thread_id
#       ↓
# Same checkpointed state
#       ↓
# Graph can remember previous state


# Different thread:
#
#     thread_id="user-456"
#
# gets separate memory.


# ============================================================
# PART 5 — TOOL CALLING IN LANGGRAPH
# ============================================================

from langchain_core.tools import tool


@tool
def get_weather(city: str):
    """Get the weather for a city."""
    return f"It is sunny in {city}."


# Typical LangGraph tool architecture:
#
#
#                  ┌─────────────┐
#                  │    LLM      │
#                  └──────┬──────┘
#                         │
#                    tool call?
#                    /         \
#                  YES          NO
#                   │            │
#                   ▼            ▼
#                 TOOL          END
#                   │
#                   ▼
#               Tool result
#                   │
#                   └──────────→ LLM
#
#
# The graph controls this loop.


# In modern LangGraph you can use ToolNode:

from langgraph.prebuilt import ToolNode


tools = [get_weather]

tool_node = ToolNode(tools)


# The ToolNode:
#
#     receives AIMessage containing tool_calls
#             ↓
#     executes appropriate tools
#             ↓
#     returns ToolMessage(s)


# ============================================================
# PART 6 — REACT AGENT
# ============================================================

from langgraph.prebuilt import create_react_agent


# ReAct:
#
#     REASON
#       ↓
#     ACT
#       ↓
#     OBSERVE
#       ↓
#     REASON
#       ↓
#     ACT
#       ↓
#     ...
#
#
# Example:
#
# User:
#     "What's the weather?"
#
# LLM:
#     I need weather information.
#
# Action:
#     get_weather("Kolkata")
#
# Observation:
#     Sunny
#
# LLM:
#     Final answer.


react_agent = create_react_agent(
    model,
    tools=[get_weather]
)


result = react_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?"
            }
        ]
    }
)


print(result)


# IMPORTANT:
#
# create_react_agent is a prebuilt implementation of the
# ReAct-style agent loop.
#
# Instead of manually constructing:
#
#     LLM node
#     ↓
#     conditional edge
#     ↓
#     ToolNode
#     ↓
#     LLM
#
# you can use the prebuilt agent.


# ============================================================
# PART 7 — HITL
# ============================================================

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


# HITL = Human In The Loop
#
# Used when the AI wants to perform an action that requires
# human approval.
#
#
# Example:
#
#     User
#       ↓
#     LLM
#       ↓
#     "I want to send this email."
#       ↓
#     INTERRUPT
#       ↓
#     HUMAN
#       ↓
#     approve / edit / reject
#       ↓
#     continue


# Modern LangGraph HITL can use an interrupt.

from langgraph.types import interrupt


def send_email_node(state):
    approval = interrupt(
        {
            "question": "Send this email?",
            "email": state["email"]
        }
    )

    if approval == "yes":
        return {
            "status": "Email sent"
        }

    return {
        "status": "Email cancelled"
    }


# Human resumes the graph:

graph.invoke(
    Command(
        resume="yes"
    ),
    config=config
)


# ------------------------------------------------------------
# HITL THREE CORE ACTIONS
# ------------------------------------------------------------

#
# 1. APPROVE
#
#     AI proposed:
#
#     send_email(...)
#
#     HUMAN:
#         approve
#
#     ↓
#
#     execute exactly that action.
#
#
# 2. EDIT
#
#     AI proposed:
#
#     send_email(
#         to="wrong@email.com"
#     )
#
#     HUMAN:
#         edit
#
#     ↓
#
#     send_email(
#         to="correct@email.com"
#     )
#
#     modified action executes.
#
#
# 3. REJECT
#
#     AI proposed:
#
#     delete_file(...)
#
#     HUMAN:
#         reject
#
#     ↓
#
#     tool/action does NOT execute.
#
#
# Memory/checkpointing is important because the graph needs
# to pause and later resume from exactly where it stopped.


# ============================================================
# PART 8 — MCP
# ============================================================

# MCP = Model Context Protocol
#
# The important idea:
#
# MCP standardizes how an AI application discovers and
# interacts with external tools/resources/prompts.
#
#
# Instead of:
#
#     AI application
#          ↓
#     custom API integration
#
# MCP gives:
#
#     AI application
#          ↓
#         MCP
#          ↓
#     MCP server
#          ↓
#       tools/data
#
#
# MCP SERVER:
#     exposes tools/resources/prompts
#
# MCP CLIENT:
#     connects to MCP server
#
# AI application:
#     uses the discovered capabilities.


# ============================================================
# PART 9 — FASTMCP SERVER — STDIO
# ============================================================

# File:
#
# weather_server.py


from fastmcp import FastMCP


mcp = FastMCP("Weather Server")


@mcp.tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is sunny in {city}."


if __name__ == "__main__":
    mcp.run(
        transport="stdio"
    )


# STDIO ARCHITECTURE:
#
#
#     MCP CLIENT
#         │
#         │ stdin/stdout
#         ▼
#     FastMCP Server
#         │
#         ▼
#     get_weather()
#
#
# No HTTP server.
#
# The client launches the MCP server as a subprocess and
# communicates through standard input/output.


# ============================================================
# PART 10 — FASTMCP SERVER — HTTP
# ============================================================

# Same server can expose HTTP transport.


mcp = FastMCP("Weather HTTP Server")


@mcp.tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is sunny in {city}."


if __name__ == "__main__":
    mcp.run(
        transport="http",
        host="127.0.0.1",
        port=8000
    )


# HTTP ARCHITECTURE:
#
#
#     MCP CLIENT
#         │
#         │ HTTP
#         ▼
#     http://127.0.0.1:8000
#         │
#         ▼
#     FastMCP Server
#         │
#         ▼
#     get_weather()
#
#
# Difference:
#
# STDIO:
#
#     local process communication
#
#
# HTTP:
#
#     network communication
#
#
# The MCP protocol remains the same idea;
# the transport is different.


# ============================================================
# PART 11 — MCP CLIENT
# ============================================================

# A client connects to an MCP server and discovers its tools.
#
# Conceptually:
#
#
#     MCP Client
#         │
#         │ connect
#         ▼
#     MCP Server
#         │
#         ▼
#     list_tools()
#         │
#         ▼
#     get_weather
#
#
# The client doesn't need to hard-code every tool's
# implementation.
#
# It can discover capabilities exposed by the server.


# Typical FastMCP client pattern:

from fastmcp import Client


async def main():

    async with Client(
        "weather_server.py"
    ) as client:

        tools = await client.list_tools()

        print(tools)

        result = await client.call_tool(
            "get_weather",
            {
                "city": "Kolkata"
            }
        )

        print(result)


# STDIO client:
#
#     Client("weather_server.py")
#
#
# HTTP client:
#
#     Client("http://127.0.0.1:8000/mcp")


# ============================================================
# PART 12 — MULTI-SERVER MCP CLIENT
# ============================================================

# This is where MCP becomes particularly useful.
#
# Your application can connect to MULTIPLE MCP servers.
#
#
#                    AI APPLICATION
#                          │
#                    MCP CLIENT
#                          │
#             ┌────────────┼────────────┐
#             │            │            │
#             ▼            ▼            ▼
#        Weather MCP    GitHub MCP   Database MCP
#          Server         Server        Server
#             │            │            │
#        weather()      search()      query()
#
#
# The application gets tools from multiple independent
# servers.


# Conceptual configuration:

servers = {
    "weather": {
        "transport": "stdio",
        "command": "python",
        "args": ["weather_server.py"],
    },

    "github": {
        "transport": "http",
        "url": "http://localhost:9000/mcp",
    },

    "database": {
        "transport": "http",
        "url": "http://localhost:8000/mcp",
    },
}


# The client connects to each server,
# discovers its tools,
# and aggregates them for the application.
#
#
# RESULTING TOOL SET:
#
#     get_weather()
#     search_github()
#     query_database()
#
#
# The LLM can then choose among tools from multiple
# MCP servers.


# ============================================================
# PART 13 — LANGGRAPH + MCP
# ============================================================

# This is the important final architecture.
#
#
#                    USER
#                     │
#                     ▼
#                  LANGGRAPH
#                     │
#                     ▼
#                    LLM
#                     │
#               "I need a tool"
#                     │
#                     ▼
#                MCP CLIENT
#                     │
#          ┌──────────┼──────────┐
#          │          │          │
#          ▼          ▼          ▼
#       Weather     GitHub    Database
#        MCP         MCP        MCP
#       Server      Server     Server
#          │          │          │
#          └──────────┼──────────┘
#                     │
#                 Tool result
#                     │
#                     ▼
#                    LLM
#                     │
#                     ▼
#                   USER
#
#
# LangGraph manages:
#
#     state
#     graph execution
#     memory
#     tool loop
#     branching
#     HITL
#
#
# MCP manages:
#
#     standardized external tool integration
#     tool discovery
#     server/client communication
#     multiple external tool servers


# ============================================================
# PART 14 — THE BIG PICTURE
# ============================================================

#
#
#                 ┌──────────────────┐
#                 │      USER        │
#                 └────────┬─────────┘
#                          │
#                          ▼
#                 ┌──────────────────┐
#                 │    LANGGRAPH     │
#                 │                  │
#                 │     State        │
#                 │     Nodes        │
#                 │     Edges        │
#                 │     Memory       │
#                 │     HITL         │
#                 └────────┬─────────┘
#                          │
#                          ▼
#                 ┌──────────────────┐
#                 │       LLM        │
#                 └────────┬─────────┘
#                          │
#                    tool call
#                          │
#                          ▼
#                 ┌──────────────────┐
#                 │    MCP CLIENT    │
#                 └────────┬─────────┘
#                          │
#             ┌────────────┼────────────┐
#             │            │            │
#             ▼            ▼            ▼
#          MCP Server   MCP Server   MCP Server
#          Weather      GitHub       Database
#             │            │            │
#             └────────────┼────────────┘
#                          │
#                       result
#                          │
#                          ▼
#                         LLM
#                          │
#                          ▼
#                         USER
#
#
# ============================================================
# PART 15 — REVISION CHEAT SHEET
# ============================================================

#
# LANGGRAPH
#
# StateGraph(State)
#     ↓
# define graph state
#
#
# add_node()
#     ↓
# add computation to graph
#
#
# add_edge()
#     ↓
# define execution path
#
#
# START
#     ↓
# graph entry point
#
#
# END
#     ↓
# graph exit point
#
#
# STATE
#     ↓
# shared data flowing through graph
#
#
# REDUCER
#     ↓
# tells LangGraph how to combine old + new state values
#
#
# CHECKPOINTER
#     ↓
# saves graph state
#
#
# thread_id
#     ↓
# identifies a conversation/execution thread
#
#
# TOOL NODE
#     ↓
# executes tools requested by the LLM
#
#
# REACT
#     ↓
# reason → act → observe → repeat
#
#
# HITL
#     ↓
# pause graph for human decision
#
#     approve
#     edit
#     reject
#
#
# MCP SERVER
#     ↓
# exposes tools/resources/prompts
#
#
# FASTMCP
#     ↓
# easy Python framework for creating MCP servers
#
#
# STDIO
#     ↓
# local process communication
#
#
# HTTP
#     ↓
# network communication
#
#
# MCP CLIENT
#     ↓
# connects to MCP server + discovers/calls capabilities
#
#
# MULTI-SERVER MCP
#     ↓
# one application/client connects to many MCP servers
#
#
# ============================================================
# PART 16 — THE DIFFERENCE BETWEEN LANGGRAPH AND MCP
# ============================================================

#
# LANGGRAPH:
#
#     "How does my AI application THINK and EXECUTE?"
#
#     State
#     Nodes
#     Edges
#     Loops
#     Memory
#     HITL
#     Agent execution
#
#
# MCP:
#
#     "How does my AI application CONNECT TO EXTERNAL
#      CAPABILITIES?"
#
#     Tools
#     Resources
#     Prompts
#     Servers
#     Clients
#     Standardized communication
#
#
# Therefore:
#
#     LangGraph ≠ MCP
#
# They solve different problems.
#
#
# LangGraph can USE MCP tools.
#
# MCP doesn't replace LangGraph's state machine.
#
#
# ============================================================
# FINAL MEMORY TRICK
# ============================================================

#
# LANGGRAPH
#
#     GRAPH
#       ↓
#     STATE
#       ↓
#     NODES
#       ↓
#     EDGES
#       ↓
#     LOOP
#       ↓
#     TOOLS
#       ↓
#     MEMORY
#       ↓
#     HITL
#
#
# MCP
#
#     CLIENT
#       ↓
#     SERVER
#       ↓
#     TOOLS
#       ↓
#     STDIO / HTTP
#
#
# MULTI-SERVER MCP
#
#     CLIENT
#       ↓
#     ┌──────┬──────┬──────┐
#     │      │      │      │
#   SERVER SERVER SERVER SERVER
#
#
# FINAL ARCHITECTURE
#
#     USER
#       ↓
#   LANGGRAPH
#       ↓
#      LLM
#       ↓
#   MCP CLIENT
#       ↓
#   ┌───┼────┐
#   ↓   ↓    ↓
# MCP MCP  MCP
# S1  S2   S3
#   └───┼────┘
#       ↓
#     RESULT
#       ↓
#      LLM
#       ↓
#      USER
#
# ============================================================